# 02 — Fine-tune Llama 3.1 8B (Step 5)

Fine-tunes Llama 3.1 8B with **QLoRA** to classify emails, then runs the fine-tuned model on the validation and test sets and saves its answers to Google Drive — in the same format as the baselines, so `src/metrics.py` scores it in the same tables.

**Label-only fine-tuning:** the model sees exactly the same prompt and email as the baselines, and learns to answer with just `{"label": "phishing"}` or `{"label": "legitimate"}`. No explanations yet — training on explanations is a separate, later run. The training logic lives in `src/finetune.py`.

**One training set per run.** Set `TRAIN_SET` in the configuration cell. Rough times on an L4 (2 epochs, plus ~5 min of evaluation):

| `TRAIN_SET` | Emails | Training time |
|---|---|---|
| `train_500` | 500 | ~10-20 min |
| `train_1500` | 1,500 | ~30-50 min |
| `train_3000` | 3,000 | ~1-1.5 h |
| `train_full` | 8,248 | ~3-5 h |

Start with `train_3000`, then the small ones; `train_full` last.

### Before you run
1. **GPU runtime:** *Runtime → Change runtime type → L4* (or A100). A T4 works but is ~3x slower and needs `BATCH_SIZE = 2`.
2. **Hugging Face token and Llama license:** same as for the baselines (`HF_TOKEN` secret, license accepted).
3. *Runtime → Run all.* If Colab disconnects, run all cells again: training continues from the last checkpoint (saved every ~10 minutes), and evaluation skips emails already done.
4. If a cell fails with `CUDA out of memory` or *"GPU memory is still in use"*, use *Runtime → **Restart session***, then *Run all* — a crashed run keeps its memory until the session restarts.

## 1. Setup — check the GPU, get the code, install libraries

In [1]:
# Show which GPU Colab gave us. If this fails, the runtime has no GPU —
# change it under Runtime → Change runtime type.
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os, sys

# Let PyTorch grow its GPU memory pool in flexible chunks, which reduces
# out-of-memory errors. Must be set BEFORE torch is imported.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Get our project code from GitHub (or update it if already there).
REPO_DIR = "/content/llm-phishing"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q https://github.com/mihaighise/llm-phishing.git {REPO_DIR}

# Install / upgrade the libraries:
#   transformers  — loads the model and its chat template; contains Trainer
#   accelerate    — places the model on the GPU
#   bitsandbytes  — 4-bit quantization (the "Q" in QLoRA)
#   peft          — LoRA adapters (the "LoRA" in QLoRA)
!pip install -q -U transformers accelerate bitsandbytes peft

# Make `import finetune` / `import inference` work.
sys.path.insert(0, f"{REPO_DIR}/src")

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


## 2. Log in to Hugging Face
Needed to download the gated Llama model. Reads the `HF_TOKEN` secret.

In [2]:
from google.colab import userdata
from huggingface_hub import login

# userdata.get reads a Colab secret, so the token never appears in the notebook.
login(token=userdata.get("HF_TOKEN"))

## 3. Build the dataset on Colab
Same as in the baseline notebook: download the Zenodo files and run `build_dataset.py`. The fingerprint check proves the training, validation and test files are identical to the ones built on the Mac.

In [3]:
import hashlib
from pathlib import Path

RAW_DIR = Path(REPO_DIR) / "data" / "zenodo"
DATA_DIR = Path(REPO_DIR) / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Download each source CSV unless it is already present.
for name in ["Nazario", "Nigerian_Fraud", "Nazario_5", "Nigerian_5", "SpamAssasin"]:
    target = RAW_DIR / f"{name}.csv"
    if not target.exists():
        url = f"https://zenodo.org/api/records/8339691/files/{name}.csv/content"
        !curl -sfL -o {target} "{url}"

# Build the splits (same command you run locally).
!python3 {REPO_DIR}/src/build_dataset.py

# Fingerprints (MD5 hashes) of the files built on the Mac.
EXPECTED = {
    "test.jsonl":       "50b428d99f00fc8bf320d827acd4aefc",
    "val.jsonl":        "8e4da4755a9e99c67e6d43d7f2ef7fc3",
    "train_500.jsonl":  "a6dd4af9735bbdf380b059a53cadc88f",
    "train_1500.jsonl": "41fe771ca2ee5d21168d7b121623a192",
    "train_3000.jsonl": "df64b0ec01a6488555b399f2319ea067",
    "train_full.jsonl": "110e083df245100a63b5bc26443896d9",
}
for fname, expected in EXPECTED.items():
    actual = hashlib.md5((DATA_DIR / fname).read_bytes()).hexdigest()
    status = "OK — identical to local build" if actual == expected else \
             "MISMATCH — data differs from the local build!"
    print(f"{fname}: {status}")

Unique usable emails: {'nazario': 1538, 'nigerian': 3286, 'legitimate': 5743}

Wrote 6 splits to /content/llm-phishing/data/processed/
split         total  phish  legit  nazario  nigerian
test           1000    500    500      300       200
val             400    200    200      120        80
train_500       500    250    250      150       100
train_1500     1500    750    750      450       300
train_3000     3000   1500   1500      900       600
train_full     8248   4124   4124     1118      3006
test.jsonl: OK — identical to local build
val.jsonl: OK — identical to local build
train_500.jsonl: OK — identical to local build
train_1500.jsonl: OK — identical to local build
train_3000.jsonl: OK — identical to local build
train_full.jsonl: OK — identical to local build


## 4. Google Drive folders
Colab deletes its files when the session ends, so everything that matters goes to Drive:
- `MyDrive/llm-phishing/finetune/<run>/checkpoints/` — saved every 50 steps (~10 min), to resume after a disconnect; only the best and the latest are kept
- `MyDrive/llm-phishing/finetune/<run>/adapter/` — the final adapter (~100-200 MB)
- `MyDrive/llm-phishing/results/raw/` — the fine-tuned model's answers, next to the baseline results

In [4]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/llm-phishing")
RESULTS_DIR = DRIVE_DIR / "results" / "raw"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 5. Configuration
Everything you might want to change is in this one cell.

In [5]:
# Make sure Python can find our modules in src/. Cell 1 already does this,
# but if Colab restarts the runtime (crash, disconnect, "Restart session"),
# that setting is forgotten and imports fail with "No module named ...".
# Re-adding it here makes this cell safe to run on its own after a restart.
import sys
SRC_DIR = "/content/llm-phishing/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import json
from prompts import PROMPT_VERSION

TRAIN_SET = "train_3000"   # "train_500", "train_1500", "train_3000" or "train_full"
EPOCHS = 2                 # passes over the training data; the best checkpoint is kept
BATCH_SIZE = 2            # training emails per GPU step; 2 on a T4 or if "CUDA out of memory"
EFFECTIVE_BATCH_SIZE = 16  # emails per weight update (see finetune.py); keep at 16
EVAL_BATCH_SIZE = 16       # emails at once when evaluating; 8 on a T4

# Name of this fine-tuned model in file names and result tables,
# e.g. "llama31_8b_ft_train_3000".
RUN_NAME = f"llama31_8b_ft_{TRAIN_SET}"
CHECKPOINT_DIR = DRIVE_DIR / "finetune" / RUN_NAME / "checkpoints"
ADAPTER_DIR = DRIVE_DIR / "finetune" / RUN_NAME / "adapter"

def load_split(name):
    """Read one split file: one email (a dict) per line."""
    with open(DATA_DIR / f"{name}.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_emails = load_split(TRAIN_SET)
val_emails = load_split("val")
test_emails = load_split("test")
print(f"{RUN_NAME}: train {len(train_emails)} / val {len(val_emails)} / "
      f"test {len(test_emails)} emails, prompt {PROMPT_VERSION}")

llama31_8b_ft_train_3000: train 3000 / val 400 / test 1000 emails, prompt v2


Preview one training example. Only the tokens marked as **graded** count towards the loss — the model is trained to write the answer, not to predict the email:

In [6]:
# Make sure Python can find our modules in src/. Cell 1 already does this,
# but if Colab restarts the runtime (crash, disconnect, "Restart session"),
# that setting is forgotten and imports fail with "No module named ...".
# Re-adding it here makes this cell safe to run on its own after a restart.
import sys
SRC_DIR = "/content/llm-phishing/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from transformers import AutoTokenizer
from finetune import build_example, IGNORE_INDEX
from inference import MODELS

# Only the tokenizer is needed here (small, downloads in seconds).
preview_tokenizer = AutoTokenizer.from_pretrained(MODELS["llama31_8b"]["hf_id"])
example = build_example(preview_tokenizer, train_emails[0])

n_graded = sum(label != IGNORE_INDEX for label in example["labels"])
graded_ids = [t for t, label in zip(example["input_ids"], example["labels"])
              if label != IGNORE_INDEX]
print(f"{len(example['input_ids'])} tokens in total, {n_graded} graded")
print("Last 300 characters of the prompt (not graded):")
prompt_ids = example["input_ids"][:len(example["input_ids"]) - n_graded]
print(repr(preview_tokenizer.decode(prompt_ids)[-300:]))
print("Graded answer:", repr(preview_tokenizer.decode(graded_ids)))

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


644 tokens in total, 8 graded
Last 300 characters of the prompt (not graded):
'bout is about 7 foot tall and presents "Animal Hospital" or something like that. tim h To unsubscribe from this group, send an email to: forteana-unsubscribe@egroups.com Your use of Yahoo! Groups is subject to http://docs.yahoo.com/info/terms/<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
Graded answer: '{"label": "legitimate"}<|eot_id|>'


## 6. Train
Loads Llama in 4-bit, adds LoRA adapters, and trains for `EPOCHS` epochs. Every 50 steps (~10 min on an L4) and at the end, it measures the loss on the 400 validation emails (~1 min) and saves a checkpoint to Drive; at the end the checkpoint with the **lowest validation loss** is kept and saved as the adapter.

The progress bar shows steps (one step = one weight update = 16 emails). `loss` is the training loss: it should drop quickly from ~1-3 towards ~0.0x. If a finished adapter already exists in Drive, training is skipped.

In [7]:
# Make sure Python can find our modules in src/. Cell 1 already does this,
# but if Colab restarts the runtime (crash, disconnect, "Restart session"),
# that setting is forgotten and imports fail with "No module named ...".
# Re-adding it here makes this cell safe to run on its own after a restart.
import sys
SRC_DIR = "/content/llm-phishing/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from inference import free_gpu_memory, check_gpu_memory
from finetune import train_adapter

if (ADAPTER_DIR / "adapter_config.json").exists():
    print(f"Adapter already trained, skipping training: {ADAPTER_DIR}")
    print("(Delete that folder in Drive to train again.)")
else:
    # Training needs an (almost) empty GPU. If a crashed run in this session
    # still holds memory, stop now with a clear message instead of failing
    # later with a confusing one.
    check_gpu_memory(max_used_gb=3)
    train_adapter(train_emails, val_emails,
                  checkpoint_dir=CHECKPOINT_DIR, adapter_dir=ADAPTER_DIR,
                  epochs=EPOCHS, batch_size=BATCH_SIZE,
                  effective_batch_size=EFFECTIVE_BATCH_SIZE)
# The training model only lived inside train_adapter(), so its memory can
# now be released before the evaluation model is loaded.
free_gpu_memory()

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

2988 examples (skipped 12 longer than 2048 tokens); tokens per example: median 754, max 2043
398 examples (skipped 2 longer than 2048 tokens); tokens per example: median 741, max 1919


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)
/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing_kwargs` is deprecated and will be removed in v6. Use `activation_checkpointing_kwargs` instead.
  warnings.warn(f"`{old}` is deprecat

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
Resuming from /content/drive/MyDrive/llm-phishing/finetune/llama31_8b_ft_train_3000/checkpoints/checkpoint-187


Epoch,Training Loss,Validation Loss
2,0.000086,0.002717


Best checkpoint: /content/drive/MyDrive/llm-phishing/finetune/llama31_8b_ft_train_3000/checkpoints/checkpoint-374
Adapter saved to /content/drive/MyDrive/llm-phishing/finetune/llama31_8b_ft_train_3000/adapter


### Validation loss during training
One row per checkpoint (every 50 steps and at the end). `eval_loss` (validation) should go down, or at least not go up. If it rises while the training loss keeps falling, the model is starting to memorise the training emails ("overfitting") — that is why the best checkpoint, not the last one, is kept.

In [8]:
import pandas as pd

history = json.loads((ADAPTER_DIR / "log_history.json").read_text())
# The log mixes training-loss entries and evaluation entries; keep one row
# per evaluation (= per checkpoint) with the most recent training loss before it.
rows, last_train_loss = [], None
for entry in history:
    if "loss" in entry:
        last_train_loss = entry["loss"]
    if "eval_loss" in entry:
        rows.append({"step": entry["step"], "epoch": round(entry["epoch"], 2),
                     "train_loss": last_train_loss,
                     "eval_loss": round(entry["eval_loss"], 4)})
pd.DataFrame(rows)

,epoch,train_loss,eval_loss
0,1.0,0.000049,0.0144
1,2.0,0.000086,0.0027


## 7. Evaluate the fine-tuned model
Loads the 4-bit Llama again with the adapter on top — the same precision as the baselines — and answers the validation and test emails with the same code as the baselines (`inference.run_model`). Answers are short, so this is fast (~1-2 min for validation, ~3-5 min for test on an L4).

Results go to `results/raw/<RUN_NAME>_<split>_all_<prompt version>.jsonl` in Drive.

In [9]:
# Make sure Python can find our modules in src/. Cell 1 already does this,
# but if Colab restarts the runtime (crash, disconnect, "Restart session"),
# that setting is forgotten and imports fail with "No module named ...".
# Re-adding it here makes this cell safe to run on its own after a restart.
import sys
SRC_DIR = "/content/llm-phishing/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from inference import load_model, free_gpu_memory, check_gpu_memory, run_model

model = tokenizer = None
free_gpu_memory()
# Loading the 4-bit model needs ~6 GB; stop with a clear message if a
# crashed run in this session still holds the GPU.
check_gpu_memory(min_free_gb=8)

model, tokenizer = load_model("llama31_8b", adapter_path=str(ADAPTER_DIR))
all_results = {}
for split, emails in [("val", val_emails), ("test", test_emails)]:
    out_path = RESULTS_DIR / f"{RUN_NAME}_{split}_all_{PROMPT_VERSION}.jsonl"
    # max_new_tokens=20: the answer is ~8 tokens; a short cap keeps a
    # misbehaving model from rambling for hundreds of tokens.
    all_results[split] = run_model(RUN_NAME, model, tokenizer, emails, out_path,
                                   batch_size=EVAL_BATCH_SIZE, max_new_tokens=20)
del model, tokenizer
free_gpu_memory()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


[llama31_8b_ft_train_3000] 0 already done, 400 to run
[llama31_8b_ft_train_3000] 48/400  (2.85 emails/s, ~2 min left)
[llama31_8b_ft_train_3000] 80/400  (2.60 emails/s, ~2 min left)
[llama31_8b_ft_train_3000] 128/400  (2.34 emails/s, ~2 min left)
[llama31_8b_ft_train_3000] 160/400  (2.24 emails/s, ~2 min left)
[llama31_8b_ft_train_3000] 208/400  (2.18 emails/s, ~1 min left)
[llama31_8b_ft_train_3000] 240/400  (2.12 emails/s, ~1 min left)
[llama31_8b_ft_train_3000] 288/400  (2.04 emails/s, ~1 min left)
[llama31_8b_ft_train_3000] 320/400  (1.95 emails/s, ~1 min left)
[llama31_8b_ft_train_3000] 368/400  (1.68 emails/s, ~0 min left)
[llama31_8b_ft_train_3000] 400/400  (1.57 emails/s, ~0 min left)
[llama31_8b_ft_train_3000] 0 already done, 1000 to run
[llama31_8b_ft_train_3000] 112/1000  (2.77 emails/s, ~5 min left)
[llama31_8b_ft_train_3000] 208/1000  (2.58 emails/s, ~5 min left)
[llama31_8b_ft_train_3000] 304/1000  (2.45 emails/s, ~5 min left)
[llama31_8b_ft_train_3000] 400/1000  (2.34 em

## 8. Quick check
Strict scoring, as in the main results: an unparseable answer counts as wrong. For comparison, on the test set: Naive Bayes 98.6%, Qwen 92.2%, Mistral 89.3%, Llama (not fine-tuned) 87.7%.

In [10]:
rows = []
for split, results in all_results.items():
    df = pd.DataFrame(results)
    correct = df["parse_ok"] & (df["label"] == df["true_label"])
    rows.append({
        "split": split,
        "emails": len(df),
        "parse_rate": df["parse_ok"].mean(),
        "accuracy": correct.mean(),
        # legitimate emails called phishing / phishing emails not caught
        "false_alarms": ((df["label"] == "phishing") & (df["true_label"] == "legitimate")).sum(),
        "missed_phishing": ((df["true_label"] == "phishing") & ~correct).sum(),
    })
pd.DataFrame(rows).round(3)

,split,emails,parse_rate,accuracy,false_alarms,missed_phishing
0,val,400,1.0,0.995,2,0
1,test,1000,1.0,0.998,1,1


## Next
1. Download `<RUN_NAME>_test_all_<prompt version>.jsonl` from `MyDrive/llm-phishing/results/raw/` into `results/raw/` on your Mac and run `python3 src/metrics.py` — the fine-tuned model appears in every table next to the baselines.
2. Learning curve: set `TRAIN_SET` to the next size and *Run all* again. Each size is its own run with its own adapter folder, so nothing is overwritten.

**Do not** change settings (epochs, learning rate, ...) based on test results — only on validation results.